# CMIE EDA for DOLPHIN


This notebook is a self-contained audit of the CMIE panel used by DOLPHIN. It explains the panel structure, the target variable, the explanatory variables, the amount of temporal coverage, and the trajectory diagnostics used to validate the discovered subgroups. A reader who has not seen the raw data should be able to understand what the dataset contains and why it is suitable for longitudinal subgroup discovery.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO = Path.cwd()
if not (REPO / "transition_subgroup_discovery").exists():
    for parent in Path.cwd().parents:
        if (parent / "transition_subgroup_discovery").exists():
            REPO = parent
            break

SRC = REPO / "transition_subgroup_discovery" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tsd.features import apply_target, build_temporal_features
from tsd.io import ensure_dir, load_json
from tsd.preprocessing import preprocess_data

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


## Dataset Role

Each row is a household-month observation. The entity is the household, identified by `HH_ID`. The time variable is `MONTH_SLOT_DATE`. The target is `TOT_INC`, which is total household income in that month. DOLPHIN searches for rules over household covariates and income-composition variables, then checks whether the households satisfying each rule have unusual total-income trajectories.


In [ ]:
CONFIG_PATH = REPO / "transition_subgroup_discovery" / "configs" / "cmie_income.json"
cfg = load_json(CONFIG_PATH)
OUT_DIR = ensure_dir(REPO / "transition_subgroup_discovery" / "outputs" / "cmie_income" / "eda")

data_path = REPO / cfg["data"]["path"]
raw = pd.read_csv(data_path)
raw["MONTH_SLOT_DATE"] = pd.to_datetime(raw["MONTH_SLOT_DATE"], errors="coerce")
raw["TOT_INC"] = pd.to_numeric(raw["TOT_INC"], errors="coerce")

print("Data path:", data_path)
print("Rows:", len(raw))
print("Households:", raw["HH_ID"].nunique())
print("Date range:", raw["MONTH_SLOT_DATE"].min(), "to", raw["MONTH_SLOT_DATE"].max())
print("Monthly periods:", raw["MONTH_SLOT_DATE"].nunique())
print("Output directory:", OUT_DIR)
raw.head()


## Variable Dictionary


In [ ]:
variable_dictionary = pd.DataFrame(
    [
        ("HH_ID", "Entity identifier", "Household ID used to link monthly observations into a trajectory."),
        ("MONTH_SLOT_DATE", "Time index", "Monthly observation date."),
        ("TOT_INC", "Target", "Total household income. This is the longitudinal target whose trajectory DOLPHIN evaluates."),
        ("wage_share", "Rule covariate", "Share of total income coming from wages."),
        ("biz_share", "Rule covariate", "Share of total income coming from business income."),
        ("transfer_share", "Rule covariate", "Share of total income coming from transfers."),
        ("capital_income_share", "Rule covariate", "Share of total income coming from capital income."),
        ("TOT_N", "Rule covariate", "Household size."),
        ("EMPLOYED_N", "Rule covariate", "Number of employed household members."),
        ("MAX_EDU_LEVEL", "Rule covariate", "Ordinal encoding of maximum education level in the household."),
        ("dominant_source_*", "Static rule covariate", "One-hot indicators for the household's dominant income source category."),
    ],
    columns=["column", "role", "meaning"],
)
variable_dictionary.to_csv(OUT_DIR / "cmie_variable_dictionary.csv", index=False)
variable_dictionary


The first table gives a compact dataset summary. The target is skewed, so the median and upper quantiles are more informative than the mean alone.


In [ ]:
summary = pd.DataFrame({
    "metric": [
        "rows",
        "households",
        "monthly_periods",
        "target_nonmissing",
        "target_mean",
        "target_median",
        "target_p90",
        "target_p99",
    ],
    "value": [
        len(raw),
        raw["HH_ID"].nunique(),
        raw["MONTH_SLOT_DATE"].nunique(),
        raw["TOT_INC"].notna().sum(),
        raw["TOT_INC"].mean(),
        raw["TOT_INC"].median(),
        raw["TOT_INC"].quantile(0.90),
        raw["TOT_INC"].quantile(0.99),
    ],
})
summary.to_csv(OUT_DIR / "cmie_panel_summary.csv", index=False)
summary


## Panel Coverage


A longitudinal method needs repeated observations for the same entity. The next tables and plots show how many months are available per household and how many households are observed in each month. Large drops in either plot would indicate that trajectory estimates may be driven by sparse panel coverage.


In [ ]:
coverage = raw.groupby("HH_ID").agg(
    n_months=("MONTH_SLOT_DATE", "nunique"),
    first_month=("MONTH_SLOT_DATE", "min"),
    last_month=("MONTH_SLOT_DATE", "max"),
    target_nonmissing=("TOT_INC", lambda s: s.notna().sum()),
    mean_income=("TOT_INC", "mean"),
    start_income=("TOT_INC", "first"),
    end_income=("TOT_INC", "last"),
)
coverage["target_change"] = coverage["end_income"] - coverage["start_income"]
coverage.to_csv(OUT_DIR / "cmie_entity_coverage.csv")
coverage.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(coverage["n_months"], bins=40, ax=axes[0], color="#2878B5")
axes[0].axvline(coverage["n_months"].median(), color="black", linestyle="--", linewidth=1.2)
axes[0].set_title("Observed Months per Household")
axes[0].set_xlabel("Number of monthly observations")

monthly_counts = raw.groupby("MONTH_SLOT_DATE")["HH_ID"].nunique()
monthly_counts.plot(ax=axes[1], color="#D55E00", linewidth=2)
axes[1].set_title("Households Observed by Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Households")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_panel_coverage.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
coverage_quantiles = coverage["n_months"].quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]).to_frame("observed_months")
coverage_quantiles.to_csv(OUT_DIR / "cmie_panel_coverage_quantiles.csv")
coverage_quantiles


## Target Distribution and Temporal Pattern


`TOT_INC` is the target trajectory. The histogram shows the cross-sectional income distribution after clipping the most extreme 1% for readability. The time series plot shows whether the target has a population-level drift over the panel.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(raw["TOT_INC"].clip(upper=raw["TOT_INC"].quantile(0.99)), bins=60, ax=axes[0], color="#009E73")
axes[0].set_title("Total Income Distribution, Clipped at p99")
axes[0].set_xlabel("Total household income")

monthly_target = raw.groupby("MONTH_SLOT_DATE")["TOT_INC"].agg(["median", "mean", "count"])
monthly_target[["median", "mean"]].plot(ax=axes[1], linewidth=2)
axes[1].set_title("Monthly Total Income")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Income")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_target_distribution_and_time.png", dpi=300, bbox_inches="tight")
plt.show()

monthly_target.to_csv(OUT_DIR / "cmie_monthly_target_summary.csv")
monthly_target.tail()


## Example Household Trajectories


In [ ]:
eligible = coverage.loc[coverage["n_months"] >= coverage["n_months"].quantile(0.75)].copy()
examples = eligible.sort_values("mean_income").iloc[
    np.linspace(0, max(len(eligible) - 1, 0), num=min(8, len(eligible)), dtype=int)
].index
example_panel = raw[raw["HH_ID"].isin(examples)].copy()

fig, ax = plt.subplots(figsize=(12, 6))
for hh_id, group in example_panel.groupby("HH_ID"):
    group = group.sort_values("MONTH_SLOT_DATE")
    ax.plot(group["MONTH_SLOT_DATE"], group["TOT_INC"], marker="o", linewidth=1.5, alpha=0.8, label=str(hh_id))
ax.set_title("Example Household Total-Income Trajectories")
ax.set_xlabel("Month")
ax.set_ylabel("Total household income")
ax.legend(title="HH_ID", fontsize=8, frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_example_household_trajectories.png", dpi=300, bbox_inches="tight")
plt.show()


## Covariates Used by DOLPHIN


The rule language uses income composition and household characteristics. The next outputs show missingness and marginal distributions for the main covariates. This helps identify variables that may dominate rules because of scale, sparsity, or strong concentration near zero.


In [ ]:
feature_cols = [
    "wage_share",
    "biz_share",
    "transfer_share",
    "capital_income_share",
    "EMPLOYED_N",
    "TOT_N",
    "MAX_EDU_LEVEL",
]
feature_cols = [c for c in feature_cols if c in raw.columns]
missing = raw[feature_cols].isna().mean().sort_values(ascending=False).to_frame("missing_fraction")
missing.to_csv(OUT_DIR / "cmie_covariate_missingness.csv")
missing


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_cols = [c for c in ["wage_share", "biz_share", "transfer_share", "capital_income_share"] if c in raw.columns]
for ax, col in zip(axes.ravel(), plot_cols):
    sns.histplot(raw[col].dropna(), bins=40, ax=ax)
    ax.set_title(col.replace("_", " "))
for ax in axes.ravel()[len(plot_cols):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_income_share_distributions.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
categorical_like = [c for c in raw.columns if c.startswith("dominant_source_") or c.startswith("income_quintile_")]
if categorical_like:
    static_rates = raw.groupby("HH_ID")[categorical_like].max().mean().sort_values(ascending=False).to_frame("share_of_households")
    static_rates.to_csv(OUT_DIR / "cmie_static_indicator_distribution.csv")
    display(static_rates)
else:
    print("No static one-hot categorical indicators found.")


## Missingness by Month


In [ ]:
monthly_missing = raw.groupby("MONTH_SLOT_DATE")[feature_cols + ["TOT_INC"]].apply(lambda frame: frame.isna().mean())
monthly_missing.to_csv(OUT_DIR / "cmie_monthly_missingness.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(monthly_missing.T, cmap="viridis", cbar_kws={"label": "Missing fraction"}, ax=ax)
ax.set_title("CMIE Missingness by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Variable")
ax.set_xticks(np.linspace(0, len(monthly_missing.index) - 1, min(8, len(monthly_missing.index))).astype(int))
ax.set_xticklabels([monthly_missing.index[i].strftime("%Y-%m") for i in ax.get_xticks().astype(int)], rotation=35, ha="right")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_monthly_missingness_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


## Lag and Window Coverage


The feature setting considers candidate lags and rolling windows every two months. We keep intervals only when at least 80% of households have enough target history. This avoids building rule features at horizons that only a minority of households can support.


In [ ]:
counts = raw.dropna(subset=["TOT_INC"]).groupby("HH_ID")["MONTH_SLOT_DATE"].nunique()
interval_rows = []
for k in range(2, 41, 2):
    interval_rows.append({
        "interval_months": k,
        "entity_target_coverage": (counts >= k + 1).mean(),
        "passes_80pct": (counts >= k + 1).mean() >= 0.8,
    })
interval_coverage = pd.DataFrame(interval_rows)
interval_coverage.to_csv(OUT_DIR / "cmie_interval_coverage.csv", index=False)
interval_coverage


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(interval_coverage["interval_months"], interval_coverage["entity_target_coverage"], marker="o")
ax.axhline(0.8, color="black", linestyle="--", linewidth=1.2)
ax.set_ylim(0, 1.05)
ax.set_title("CMIE Target-History Coverage by Interval")
ax.set_xlabel("Lag/window length in months")
ax.set_ylabel("Share of households with enough target history")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_interval_coverage.png", dpi=300, bbox_inches="tight")
plt.show()


## DOLPHIN Feature Matrix Availability


After temporal feature engineering, DOLPHIN keeps only engineered features with at least 80% availability at the entity level. This section reports how many candidate features are created and how many survive that filter.


In [ ]:
work, target_col, target_excludes = apply_target(raw, cfg["targets"][0])
table, feature_names = build_temporal_features(
    work,
    id_col=cfg["data"]["id_col"],
    date_col=cfg["data"]["date_col"],
    target_col=target_col,
    feature_cfg=cfg["feature_engineering"],
    exclude_cols=target_excludes,
)
entity_table = table.groupby(cfg["data"]["id_col"], sort=False).tail(1)
X = entity_table[feature_names].apply(pd.to_numeric, errors="coerce")
feature_missing = X.isna().mean().sort_values()
feature_report = pd.DataFrame({
    "feature": feature_missing.index,
    "missing_fraction": feature_missing.values,
    "retained_at_80pct": feature_missing.values < cfg["methods"]["trajtrack"]["max_missing_frac"],
})
feature_report.to_csv(OUT_DIR / "cmie_engineered_feature_availability.csv", index=False)
print("Engineered features:", len(feature_names))
print("Retained at 80% availability:", int(feature_report["retained_at_80pct"].sum()))
feature_report.head(20)


## Trajectory Diagnostics


These diagnostics summarize the target trajectories before subgroup discovery. `baseline_deviation` measures how far a household's mean-centered trajectory is from the global mean-centered trajectory. `mean_abs_monthly_change` measures how much the household's income changes from month to month on average.


In [ ]:
pivot = raw.pivot_table(index="HH_ID", columns="MONTH_SLOT_DATE", values="TOT_INC", aggfunc="mean")
valid = pivot.dropna(thresh=6)
interp = valid.interpolate(axis=1, limit_direction="both")
values = interp.to_numpy(dtype=float)
centered = values - values.mean(axis=1, keepdims=True)
baseline = centered.mean(axis=0)
baseline_deviation = np.sqrt(((centered - baseline) ** 2).sum(axis=1))
mean_abs_change = np.mean(np.abs(np.diff(values, axis=1)), axis=1)
diag = pd.DataFrame({
    "HH_ID": interp.index,
    "baseline_deviation": baseline_deviation,
    "mean_abs_monthly_change": mean_abs_change,
    "mean_income": values.mean(axis=1),
    "start_income": values[:, 0],
    "end_income": values[:, -1],
})
diag.to_csv(OUT_DIR / "cmie_trajectory_diagnostics.csv", index=False)
diag.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(diag["baseline_deviation"].clip(upper=diag["baseline_deviation"].quantile(0.99)), bins=60, ax=axes[0], color="#CC79A7")
axes[0].set_title("Trajectory Deviation from Global Baseline")
axes[0].set_xlabel("Euclidean deviation, clipped at p99")

sns.histplot(diag["mean_abs_monthly_change"].clip(upper=diag["mean_abs_monthly_change"].quantile(0.99)), bins=60, ax=axes[1], color="#0072B2")
axes[1].set_title("Mean Absolute Month-to-Month Income Change")
axes[1].set_xlabel("Mean absolute monthly change, clipped at p99")
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_trajectory_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()


## Result Validation: Do DOLPHIN Rules Select Distinct Trajectories?


This section links the EDA back to the discovered DOLPHIN rules. It checks whether the selected subgroups differ from the full population on trajectory diagnostics. A useful rule should generally show higher trajectory deviation, higher dynamic change, or a clearly different mean-centered trajectory shape.


In [ ]:
RESULT_DIR = REPO / "transition_subgroup_discovery" / "outputs" / "cmie_income" / "total_income" / "trajtrack"
rules = pd.read_csv(RESULT_DIR / "rules.csv")
membership = pd.read_csv(RESULT_DIR / "membership.csv")

display_cols = [
    "rank",
    "rule",
    "n_entities",
    "support",
    "baseline_effect_size",
    "subgroup_trend_change",
    "trend_change_difference",
    "relative_trend_direction",
]
rules[display_cols].to_csv(OUT_DIR / "cmie_dolphin_rule_validation_table.csv", index=False)
rules[display_cols]


In [ ]:
diag_valid = diag.merge(membership, left_on="HH_ID", right_on="entity", how="left")
rule_cols = [c for c in membership.columns if c.startswith("rule_")]
records = []
for col in rule_cols:
    rank = int(col.split("_")[1])
    mask = diag_valid[col].fillna(0).astype(bool)
    complement = ~mask
    for metric in ["baseline_deviation", "mean_abs_monthly_change"]:
        records.append({
            "rule": f"Rule {rank}",
            "metric": metric,
            "subgroup_median": diag_valid.loc[mask, metric].median(),
            "population_median": diag_valid[metric].median(),
            "complement_median": diag_valid.loc[complement, metric].median(),
            "median_ratio_to_population": diag_valid.loc[mask, metric].median() / max(diag_valid[metric].median(), 1e-12),
            "n": int(mask.sum()),
        })
validation_summary = pd.DataFrame(records)
validation_summary.to_csv(OUT_DIR / "cmie_dolphin_subgroup_validation_summary.csv", index=False)
validation_summary


In [ ]:
plot_rows = []
for col in rule_cols:
    rank = int(col.split("_")[1])
    mask = diag_valid[col].fillna(0).astype(bool)
    subgroup = diag_valid.loc[mask, ["baseline_deviation", "mean_abs_monthly_change"]].copy()
    subgroup["group"] = f"Rule {rank}"
    plot_rows.append(subgroup)
population_sample = diag_valid[["baseline_deviation", "mean_abs_monthly_change"]].sample(
    n=min(3000, len(diag_valid)), random_state=42
).copy()
population_sample["group"] = "Population"
plot_df = pd.concat([population_sample, *plot_rows], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(
    data=plot_df,
    x="group",
    y="baseline_deviation",
    ax=axes[0],
    showfliers=False,
    color="#8DBBD9",
)
axes[0].set_title("Baseline-Deviation Validation")
axes[0].set_xlabel("")
axes[0].set_ylabel("Deviation from global trajectory baseline")
axes[0].tick_params(axis="x", rotation=25)

sns.boxplot(
    data=plot_df,
    x="group",
    y="mean_abs_monthly_change",
    ax=axes[1],
    showfliers=False,
    color="#E6A57E",
)
axes[1].set_title("Trajectory-Change Validation")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean absolute month-to-month change")
axes[1].tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_dolphin_subgroup_validation_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
rule_colors = sns.color_palette("tab10", n_colors=max(3, len(rule_cols)))
fig, ax = plt.subplots(figsize=(11, 6))
time_axis = np.arange(values.shape[1])
iqr_low = np.quantile(centered, 0.25, axis=0)
iqr_high = np.quantile(centered, 0.75, axis=0)
ax.fill_between(time_axis, iqr_low, iqr_high, color="lightgray", alpha=0.65, label="Population IQR")
ax.plot(time_axis, baseline, color="black", linewidth=2.2, label="Global baseline")
for idx, col in enumerate(rule_cols):
    rank = int(col.split("_")[1])
    member_ids = set(membership.loc[membership[col] == 1, "entity"])
    mask = np.array([hh in member_ids for hh in interp.index])
    if mask.sum() == 0:
        continue
    ax.plot(time_axis, centered[mask].mean(axis=0), linewidth=2.4, color=rule_colors[idx], label=f"Rule {rank} (n={mask.sum()})")
ax.set_title("DOLPHIN Rule Trajectories on CMIE Total Income")
ax.set_xlabel("Aligned trajectory grid")
ax.set_ylabel("Mean-centered total income")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "cmie_dolphin_rule_trajectory_overlay.png", dpi=300, bbox_inches="tight")
plt.show()
